# 09 · Final test (run5: Juice Shop + DVWA)

This is the locked test. **run5** was generated once (seed 5) on two sites, after the code was frozen, and nobody looked at it. DVWA is a different site from the one we built and tuned on, so it is the strongest evidence that the detector works on a site it has never seen.

**Rules for this notebook:**
- The code is frozen first (git tag), and the commit is printed below.
- With `RUN_FINAL = False` it runs on development data (run3, run4 and the small DVWA check run), to test the notebook itself. Dev files that are not on this machine are skipped.
- The SHA-256 of every file read is printed first. In the final run it is compared with `run5_hashes.txt`, written when run5 was generated; if any file changed since then, the notebook stops before any result is shown.
- `RUN_FINAL = True` is run **once**. Whatever comes out goes in the report. Nothing is changed after seeing it.

run4 was the first locked test. It showed that the session layer missed the brute force and the ID enumeration on DVWA (0%), because it assumed the way Juice Shop logs a login and an ID. That was fixed after seeing run4, so run4 became development data and run5 replaced it.

The full pipeline is the same as in the backend: the detector per request, the session layer (brute force, enumeration, scan) and the attack type. The rules-only baseline is shown with and without the same session layer, so the table shows what the model itself adds.

In [1]:
RUN_FINAL = False   # set to True ONCE, after the code is frozen

In [2]:
import hashlib, re, subprocess, sys
from pathlib import Path, PureWindowsPath
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from eval.final_test import recall_by_type, run, summary, type_accuracy

commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=ROOT, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(['git', 'status', '--porcelain', '--', 'ml', 'eval', 'backend'], cwd=ROOT,
                       capture_output=True, text=True).stdout.strip()
print('code commit:', commit, '(uncommitted changes in ml/eval/backend!)' if dirty else '(clean)')

D = ROOT / 'data'
if RUN_FINAL:
    RUNS = {'Juice Shop (run5)': (D / 'raw' / 'juiceshop_generated_run5.log', D / 'labels' / 'requests_run5.csv'),
            'DVWA (run5)': (D / 'raw' / 'dvwa_generated_run5.log', D / 'labels' / 'requests_dvwa_run5.csv')}
else:
    RUNS = {'Juice Shop (run3, dev)': (D / 'raw' / 'juiceshop_generated_run3.log', D / 'labels' / 'requests_run3.csv'),
            'Juice Shop (run4, dev)': (D / 'raw' / 'juiceshop_generated_run4.log', D / 'labels' / 'requests_run4.csv'),
            'DVWA (run4, dev)': (D / 'raw' / 'dvwa_generated_run4.log', D / 'labels' / 'requests_dvwa_run4.csv'),
            'DVWA (check run, dev)': (D / 'raw' / 'dvwa_generated_dvwa_check.log', D / 'labels' / 'requests_dvwa_dvwa_check.csv')}
    for name, files in list(RUNS.items()):
        if not all(f.exists() for f in files):
            print(f'{name}: skipped, files not on this machine')
            del RUNS[name]

# Fingerprint of every file read. In the final run it must match run5_hashes.txt, written when
# run5 was generated; if any file changed since then, stop before any result is shown.
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest().upper()   # upper: same as Get-FileHash

hashes = {f.name: sha256(f) for files in RUNS.values() for f in files}
for file_name, h in hashes.items():
    print(h, file_name)
if RUN_FINAL:
    raw = (D / 'labels' / 'run5_hashes.txt').read_bytes()   # missing: ask for it, never skip the check
    text = raw.decode('utf-16') if raw[:2] in (b'\xff\xfe', b'\xfe\xff') else raw.decode('utf-8-sig')
    expected = {PureWindowsPath(p.strip()).name: h.upper()
                for h, p in re.findall(r'\b([0-9A-Fa-f]{64})\s+(.+)', text)}
    changed = [f for f, h in hashes.items() if expected.get(f) != h]
    if changed:
        raise RuntimeError(f'Not as in run5_hashes.txt (changed, or missing from the list): {changed}. '
                           'Stop here, do not look at any result.')
    print('all files match run5_hashes.txt')

results = {}
for name, (log, labels) in RUNS.items():
    req, sessions, report = run(log, labels)
    results[name] = (req, sessions)
    print(name, report)

code commit: 9e08081 (clean)
Juice Shop (run4, dev): skipped, files not on this machine
DVWA (run4, dev): skipped, files not on this machine
DVWA (check run, dev): skipped, files not on this machine
1FDB0F1AE489242CBDF47F2950AC30A61E553681DB6F0065F423BABF4BE30CA0 juiceshop_generated_run3.log
055E3DA5909AA1BDE1F1794A4BFAC8A540B1A7A5101215749AAF144B9E58962E requests_run3.csv


Juice Shop (run3, dev) {'log_lines': 4397, 'labels': 4397, 'matched': 4397, 'unmatched_log_lines': 0, 'unmatched_labels': 0}


## 1. Requests: how many attacks are caught, and how many false alarms

In [3]:
pd.concat({name: summary(req) for name, (req, _) in results.items()})

recall  precision  false alarms  \
                       detector                                            
Juice Shop (run3, dev) rules              0.218        1.0             0   
                       rules + sessions   0.985        1.0             0   
                       model              0.364        1.0             0   
                       model + sessions   0.997        1.0             0   

                                         normal requests  attacks  
                       detector                                    
Juice Shop (run3, dev) rules                        3328     1069  
                       rules + sessions             3328     1069  
                       model                        3328     1069  
                       model + sessions             3328     1069

## 2. Recall by attack type

In [4]:
pd.concat({name: recall_by_type(req) for name, (req, _) in results.items()})

rules  rules + sessions  model  \
                       attack_type                                   
Juice Shop (run3, dev) bruteforce   0.000             1.000  0.000   
                       cmdi         0.969             0.969  0.990   
                       enum         0.000             1.000  0.000   
                       scan         0.000             1.000  0.413   
                       sqli         0.829             0.829  0.974   
                       traversal    0.293             1.000  1.000   
                       xss          1.000             1.000  1.000   

                                    model + sessions  
                       attack_type                    
Juice Shop (run3, dev) bruteforce              1.000  
                       cmdi                    0.990  
                       enum                    1.000  
                       scan                    1.000  
                       sqli                    0.974  
                       traversal               1.000  
                       xss                     1.000

## 3. Sessions: is every attacker found, and no normal user flagged?

In [5]:
pd.concat({name: pd.crosstab(s['truth'], s['risk_level']) for name, (_, s) in results.items()})

risk_level                       high  medium  none
                       truth                       
Juice Shop (run3, dev) attacker    47      43     0
                       normal       0       0   210

## 4. Attack type of the caught payload attacks

In [6]:
for name, (req, _) in results.items():
    table, acc = type_accuracy(req)
    print(f'{name}: {acc:.1%} correct')
    display(table)

Juice Shop (run3, dev): 95.8% correct


predicted,cmdi,sqli,traversal,xss
true,,,,
cmdi,92,4,0,0
sqli,2,68,0,4
traversal,0,0,58,0
xss,0,2,0,57


## 5. Same logs without request bodies

Most real access logs do not store the body. This shows what is lost (attacks sent in a POST body) and checks that false alarms stay at zero.

In [7]:
pd.concat({name: summary(run(log, labels, drop_body=True)[0]) for name, (log, labels) in RUNS.items()})

recall  precision  false alarms  \
                       detector                                            
Juice Shop (run3, dev) rules              0.073        1.0             0   
                       rules + sessions   0.843        1.0             0   
                       model              0.209        1.0             0   
                       model + sessions   0.845        1.0             0   

                                         normal requests  attacks  
                       detector                                    
Juice Shop (run3, dev) rules                        3328     1069  
                       rules + sessions             3328     1069  
                       model                        3328     1069  
                       model + sessions             3328     1069

## 6. What this shows

*(Write this after the one final run, from the numbers above. Do not change the code or rerun.)*